In [1]:
#Bronze layer-Raw Ingestion from ADLS
#Fleet Data Platform Modernisation
#=========================================

StatementMeta(sparkpool01, 4, 2, Finished, Available, Finished, False)

In [2]:
storage_account = "stfleetcndatalakeprod"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/sftp_ingestion/"
bronze_delta_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/delta/"

print(f"Bronze source path: {bronze_path}")
print(f"Bronze delta path: {bronze_delta_path}")

StatementMeta(sparkpool01, 4, 3, Finished, Available, Finished, False)

Bronze source path: abfss://bronze@stfleetcndatalakeprod.dfs.core.windows.net/sftp_ingestion/
Bronze delta path: abfss://bronze@stfleetcndatalakeprod.dfs.core.windows.net/delta/


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Read all CSV files from bronze sftp_ingestion folder
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(bronze_path)

print(f"Raw records loaded: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(5)

StatementMeta(sparkpool01, 4, 4, Finished, Available, Finished, False)

Raw records loaded: 100
root
 |-- event_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- speed_mph: double (nullable = true)
 |-- fuel_level_pct: double (nullable = true)
 |-- engine_temperature_c: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

+---------+----------+-------------------+---------+--------------+--------------------+---------+---------+
| event_id|vehicle_id|          timestamp|speed_mph|fuel_level_pct|engine_temperature_c| latitude|longitude|
+---------+----------+-------------------+---------+--------------+--------------------+---------+---------+
|EVT-00001|    VH-005|2024-06-10 06:00:00|    14.01|         24.36|               78.89|51.309946| -2.13828|
|EVT-00002|    VH-001|2024-06-10 06:05:00|     9.41|         16.41|               96.98|51.388573|-2.122355|
|EVT-00003|    VH-003|2024-06-10 06:10:00|    16.48|         43.08|            

In [8]:
from datetime import datetime

# Add ingestion timestamp
df_bronze = df_raw.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
).withColumn(
    "source_file",
    F.input_file_name()
)

# Write as Delta format to bronze delta layer
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True)\
    .save(bronze_delta_path)

print(f"Bronze Delta written successfully")
print(f"Total records: {df_bronze.count()}")

StatementMeta(sparkpool01, 4, 9, Finished, Available, Finished, False)

Bronze Delta written successfully
Total records: 100
